In [10]:
import json
import random

import numpy as np
import tensorflow as tf
from keras.layers import TextVectorization
from tensorflow import Tensor
from tqdm import tqdm


# =============================================================================
# PATHS
# =============================================================================

CAPTIONS_FILE = "../data/captions/captions_split_clean.json"
OUTPUT_DIR = "../data/records"


# =============================================================================
# SPLIT CONFIGURATION (NEW)
# =============================================================================

TRAIN_SIZE = None      # None = usa tutto il train
VAL_SIZE = None        # modificabile
TEST_SIZE = VAL_SIZE   # obbligato uguale a VAL_SIZE

SEED = 42
random.seed(SEED)


# =============================================================================
# LOAD DATA
# =============================================================================

def load_captions(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def sample_split(split_dict: dict, max_items: int | None) -> dict:
    if max_items is None:
        return split_dict

    keys = list(split_dict.keys())
    random.shuffle(keys)

    keys = keys[:max_items]

    return {k: split_dict[k] for k in keys}


# =============================================================================
# PREPROCESSING
# =============================================================================

def compute_max_len(captions: dict[str, list[str]], percentile: int = 97) -> int:
    lengths = [
        len(c.split())
        for caps in captions.values()
        for c in caps
    ]
    return int(np.percentile(lengths, percentile))


def truncate_captions(
    captions: dict[str, list[str]],
    max_len: int,
) -> tuple[dict[str, list[str]], int]:

    out = {}
    count = 0

    for img, caps in captions.items():
        processed = []

        for c in caps:
            words = c.split()

            if len(words) > max_len:
                processed.append(" ".join(words[:max_len]))
                count += 1
            else:
                processed.append(c)

        out[img] = processed

    return out, count


def add_start_end_tokens(captions: dict[str, list[str]]) -> dict[str, list[str]]:

    return {
        img: [f"[START] {c} [END]" for c in caps]
        for img, caps in captions.items()
    }


# =============================================================================
# VECTORIZE
# =============================================================================

def create_vectorizer(
    captions: dict[str, list[str]],
    output_sequence_length: int,
) -> TextVectorization:

    vectorizer = TextVectorization(
        output_mode="int",
        split="whitespace",
        standardize=None, # type: ignore
        output_sequence_length=output_sequence_length,
        name="text_vectorization",
    )

    flat = [c for caps in captions.values() for c in caps]
    vectorizer.adapt(flat)

    return vectorizer


def vectorize_captions(
    vectorizer: TextVectorization,
    captions: dict[str, list[str]],
) -> Tensor:

    flat = [c for caps in captions.values() for c in caps]
    return vectorizer(flat)


def split_teacher_forcing(captions: Tensor) -> tuple[Tensor, Tensor]:
    return captions[:, :-1], captions[:, 1:] # type: ignore


# =============================================================================
# DATASET
# =============================================================================

def build_dataset(
    captions: dict[str, list[str]],
    input_tensor: Tensor,
    target_tensor: Tensor,
):

    dataset = []
    idx = 0

    for img, caps in tqdm(captions.items(), desc="Building dataset"):
        for _ in caps:
            dataset.append({
                "image": img,
                "input": input_tensor[idx], # type: ignore
                "target": target_tensor[idx], # type: ignore
            })
            idx += 1

    return dataset


# =============================================================================
# TFRECORD
# =============================================================================

def create_tfrecord(dataset, output_file: str) -> None:

    with tf.io.TFRecordWriter(output_file) as writer:

        for sample in tqdm(dataset, desc="Writing TFRecord"):

            example = tf.train.Example(
                features=tf.train.Features(
                    feature={
                        "image": tf.train.Feature(
                            bytes_list=tf.train.BytesList(
                                value=[sample["image"].encode()]
                            )
                        ),
                        "input": tf.train.Feature(
                            int64_list=tf.train.Int64List(
                                value=sample["input"].numpy().astype(np.int64).tolist()
                            )
                        ),
                        "target": tf.train.Feature(
                            int64_list=tf.train.Int64List(
                                value=sample["target"].numpy().astype(np.int64).tolist()
                            )
                        ),
                    }
                )
            )

            writer.write(example.SerializeToString()) # type: ignore


# =============================================================================
# PIPELINE
# =============================================================================

def process_split(
    split_name: str,
    captions: dict,
    vectorizer: TextVectorization | None,
    max_len: int,
):

    print(f"\n=== Processing {split_name} ===")

    captions = captions[split_name]

    captions, truncated = truncate_captions(captions, max_len)
    print(f"{split_name}: truncated {truncated} captions")

    captions = add_start_end_tokens(captions)

    if vectorizer is None:
        vectorizer = create_vectorizer(captions, max_len + 2) # [START] and [END] 

    tensor = vectorize_captions(vectorizer, captions)

    inp, tgt = split_teacher_forcing(tensor)

    dataset = build_dataset(captions, inp, tgt)

    create_tfrecord(dataset, f"{OUTPUT_DIR}/{split_name}_{len(captions)}.tfrecord")

    print(f"Saved: {OUTPUT_DIR}/{split_name}.tfrecord")

    return dataset, vectorizer


def save_config(max_len: int, vocab_size: int, vec_config, path: str):

    config = {
        "max_len": max_len,
        "vocab_size": vocab_size,
        "message": "adapt vec to training captions before usage",
        "vec_config": vec_config,
    }

    with open(path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)


# =============================================================================
# MAIN
# =============================================================================

captions = load_captions(CAPTIONS_FILE)

# NEW: sampling controllato
captions["train"] = sample_split(captions["train"], TRAIN_SIZE)
captions["val"] = sample_split(captions["val"], VAL_SIZE)
captions["test"] = sample_split(captions["test"], TEST_SIZE)

train_captions = captions["train"]

max_len = compute_max_len(train_captions, percentile=97)
print(f"Max length: {max_len}")

train_dataset, vectorizer = process_split(
    "train",
    captions,
    None,
    max_len,
)

vocab_size = len(vectorizer.get_vocabulary())
print(f"Vocabulary size: {vocab_size}")

vec_config = vectorizer.get_config()

val_dataset, _ = process_split(
    "val",
    captions,
    vectorizer,
    max_len,
)

test_dataset, _ = process_split(
    "test",
    captions,
    vectorizer,
    max_len,
)

print("\nTFRecords created successfully.")

save_config(
    max_len=max_len + 1, # vengono aggiunti [START] ed [END], ma poi si applica teacher forcing
    vocab_size=vocab_size,
    vec_config=vec_config,
    path="../config/config.json"
)

Max length: 19

=== Processing train ===
train: truncated 3569 captions


Writing TFRecord: 100%|██████████| 127130/127130 [00:03<00:00, 38891.58it/s]


Saved: ../data/records/train.tfrecord
Vocabulary size: 18028

=== Processing val ===
val: truncated 434 captions


Writing TFRecord: 100%|██████████| 15890/15890 [00:00<00:00, 38896.03it/s]


Saved: ../data/records/val.tfrecord

=== Processing test ===
test: truncated 404 captions


Writing TFRecord: 100%|██████████| 15895/15895 [00:00<00:00, 38833.73it/s]

Saved: ../data/records/test.tfrecord

TFRecords created successfully.
